<a href="https://colab.research.google.com/github/prasanna-venkatesh-m/SLM-Finetuning-Demo/blob/main/SLM-Finetuning-without-Unsloth.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q transformers datasets peft accelerate bitsandbytes trl
!pip install wandb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 528.8/528.8 kB 36.9 MB/s eta 0:00:00


In [2]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer

In [3]:
import wandb
wandb.login()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: prasannavenkatesh-m (prasannavenkatesh-m-arise) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [4]:
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token  # fix PAD token warning

from transformers import BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,  # FP16, NOT BF16
    bnb_4bit_use_double_quant=True
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    quantization_config=bnb_config
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [5]:
# prepare model for k-bit training
model = prepare_model_for_kbit_training(model)

# attach LoRA adapters
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj","v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

# disable caching (required for gradient checkpointing)
model.config.use_cache = False
model.gradient_checkpointing_enable()

# force FP16
model.half()

trainable params: 2,252,800 || all params: 1,102,301,184 || trainable%: 0.2044


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(32000, 2048)
        (layers): ModuleList(
          (0-21): 22 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=2048, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): Line

In [6]:
dataset = load_dataset("json", data_files="arise_finetune_dataset_2196.json")

# combine instruction, input, output into single text
def format_prompt(example):
    if example["input"].strip() != "":
        text = f"""### Instruction:
{example['instruction']}

### Input:
{example['input']}

### Response:
{example['output']}"""
    else:
        text = f"""### Instruction:
{example['instruction']}

### Response:
{example['output']}"""
    return {"text": text}

dataset = dataset.map(format_prompt)

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/2196 [00:00<?, ? examples/s]

In [7]:
def tokenize(example):
    tokens = tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=512
    )
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

dataset = dataset.map(tokenize, remove_columns=dataset["train"].column_names)

Map:   0%|          | 0/2196 [00:00<?, ? examples/s]

In [8]:
training_args = TrainingArguments(
    output_dir="./slm-finetuned",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=1,
    learning_rate=2e-4,
    logging_steps=10,
    save_steps=200,

    fp16=False,  # do NOT use bf16 or fp16 auto — manual FP16 is used
    bf16=False,

    optim="paged_adamw_8bit",
    report_to="wandb",
    run_name="tinyllama-arise-finetune"
)

In [9]:
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset["train"],
    args=training_args
)

Truncating train dataset:   0%|          | 0/2196 [00:00<?, ? examples/s]

In [10]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Step,Training Loss
10,1.519346
20,0.252384
30,0.196214
40,0.163839
50,0.140262
60,0.126897
70,0.113216
80,0.100598
90,0.086530
100,0.074237


TrainOutput(global_step=275, training_loss=0.12531937290321696, metrics={'train_runtime': 984.1273, 'train_samples_per_second': 2.231, 'train_steps_per_second': 0.279, 'total_flos': 6994134048964608.0, 'train_loss': 0.12531937290321696})

In [ ]:
trainer.model.save_pretrained("tinyllama-arise-lora")
tokenizer.save_pretrained("tinyllama-arise-lora")

('tinyllama-arise-lora/tokenizer_config.json',
 'tinyllama-arise-lora/chat_template.jinja',
 'tinyllama-arise-lora/tokenizer.json')

In [ ]:
!zip -r tinyllama-arise-lora.zip tinyllama-arise-lora

  adding: tinyllama-arise-lora/ (stored 0%)
  adding: tinyllama-arise-lora/README.md (deflated 65%)
  adding: tinyllama-arise-lora/chat_template.jinja (deflated 60%)
  adding: tinyllama-arise-lora/adapter_model.safetensors (deflated 22%)
  adding: tinyllama-arise-lora/tokenizer_config.json (deflated 46%)
  adding: tinyllama-arise-lora/tokenizer.json (deflated 85%)
  adding: tinyllama-arise-lora/adapter_config.json (deflated 57%)


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

base_model = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(base_model)

model = AutoModelForCausalLM.from_pretrained(
    base_model,
    device_map="auto",
    torch_dtype=torch.float16
)

model = PeftModel.from_pretrained(model, "tinyllama-arise-lora")

model.eval()

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(32000, 2048)
        (layers): ModuleList(
          (0-21): 22 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=2048, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): Linear(in_fe

In [ ]:
prompt = """### Instruction:
Where is the head office of Arise Capital located?

### Response:
"""

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

output = model.generate(
    **inputs,
    max_new_tokens=150,
    temperature=0.7,
    top_p=0.9
)

print(tokenizer.decode(output[0], skip_special_tokens=True))

### Instruction:
Where is the head office of Arise Capital located?

### Response:
The head office of Arise Capital is located at 1010 Turtle Point, Sector 17B, Noida, UT, India.
